# 03. Финальная мультимодальная модель: Protocol-Fixed Efficient HADES

**Цель раздела.** Провести полный исследовательский цикл для финальной модели: от предпосылок и конфигурации до обучения, анализа качества, анализа ошибок, проверки alert-level и готовности к внедрению в dashboard.

**Гипотеза.** При едином protocol-fixed контуре (`stratified_by_workload`, `workload_norm`, `raw_plus_deviation`) мультимодальная HADES-like архитектура дает устойчивое качество по горизонтам предупреждения и воспроизводимые артефакты для демонстрации.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import auc, precision_recall_curve, roc_curve

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
CONTRACT_PATH = ARTIFACT_DIR / "contract" / "strict_intensity_contract.csv.gz"
REPORT_DIR = ARTIFACT_DIR / "reports"
BASELINE_DIR = REPORT_DIR / "protocol_fixed_baselines"
MODEL_OUT_DIR = ARTIFACT_DIR / "model" / "protocol_fixed_efficient_hades"
RUN_ROOT = ARTIFACT_DIR / "model_runs" / "protocol_fixed_hades"
RUN_ID = "final_protocol_fixed_hades"
BUNDLE_DIR = ARTIFACT_DIR / "bundle"

def run_command(command: list[str]) -> None:
    print(" ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

def safe_read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def find_run_dir(run_root: Path, run_id: str, model_out_dir: Path) -> Path:
    primary = run_root / run_id
    if (primary / "comparison_metrics.csv").exists():
        return primary
    if (model_out_dir / "comparison_metrics.csv").exists():
        return model_out_dir
    return primary

def file_status_table(paths: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for key, path in paths.items():
        rows.append(
            {
                "артефакт": key,
                "путь": str(path.relative_to(PROJECT_ROOT)) if path.is_absolute() and str(path).startswith(str(PROJECT_ROOT)) else str(path),
                "найден": path.exists(),
                "размер_байт": path.stat().st_size if path.exists() and path.is_file() else np.nan,
            }
        )
    return pd.DataFrame(rows)

def prepare_target_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if "y_anomaly" not in out.columns and {"y_warning", "y_risk"}.issubset(out.columns):
        out["y_anomaly"] = np.maximum(out["y_warning"].astype(int), out["y_risk"].astype(int))
    for col in ["window_ts", "window_end"]:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], utc=True, errors="coerce")
    return out


## 1. Предпосылки и данные

**Цель раздела.** Проверить предпосылки обучения: баланс классов по target, распределение окон по split и workload, prevalence по горизонтам `30/60/120/300` секунд.


In [ ]:
contract = prepare_target_frame(safe_read_csv(CONTRACT_PATH))
if contract.empty:
    raise FileNotFoundError(f"Не найден contract: {CONTRACT_PATH}")

target_cols = [c for c in ["y_warning", "y_risk", "y_anomaly", "horizon_30s", "horizon_60s", "horizon_120s", "horizon_300s"] if c in contract.columns]
class_table = contract.groupby("split")[target_cols].mean().T.rename_axis("target").reset_index()
for col in class_table.columns[1:]:
    class_table[col] = (class_table[col] * 100).round(2)
display(class_table)


In [ ]:
if target_cols:
    fig, ax = plt.subplots(figsize=(11, 4))
    plot_df = contract.groupby("split")[target_cols].mean().T * 100
    plot_df.plot(kind="bar", ax=ax)
    ax.set_title("Доля positive-окон по split, %")
    ax.set_xlabel("Target")
    ax.set_ylabel("Positive windows, %")
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()

workload_split = contract.groupby(["workload", "split"]).size().rename("rows").reset_index()
display(workload_split.head(20))

pivot = workload_split.pivot(index="workload", columns="split", values="rows").fillna(0)
fig, ax = plt.subplots(figsize=(10, 4))
pivot.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Распределение окон по workload и split")
ax.set_xlabel("Workload")
ax.set_ylabel("Количество окон")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()


**Локальный вывод.** Баланс классов и структура workload определяют сложность задачи. Поэтому сравнение моделей дальше выполняется строго в одном protocol-fixed контуре.


## 2. Метод и конфигурация эксперимента

**Цель раздела.** Зафиксировать единственную конфигурацию финального обучения и параметры оценки.


In [ ]:
FINAL_CONFIG = {
    "run_id": RUN_ID,
    "device": "cuda",
    "split_protocol": "stratified_by_workload",
    "variant": "modern_metric",
    "seed": 43,
    "monitor_target": "horizon_60s",
    "alert_target": "horizon_60s",
    "normalization": "workload_norm",
    "metric_feature_mode": "raw_plus_deviation",
    "epochs": 30,
    "batch_size": 128,
    "patience": 8,
    "fp_budget": 0.10,
    "quantization": "dynamic",
}

cfg_table = pd.DataFrame([{"параметр": k, "значение": v} for k, v in FINAL_CONFIG.items()])
display(cfg_table)


**Локальный вывод.** Основной акцент - раннее предупреждение на `horizon_60s` при сохранении качества на горизонтах `120/300` секунд.


## 3. Полный запуск обучения и экспорт

**Цель раздела.** Выполнить полное обучение (`Run All`) и сформировать production-артефакты для dashboard.


In [ ]:
FULL_RUN = True
EXPORT_AFTER_TRAIN = True

train_command = [
    sys.executable, "-m", "dashboard.training.final_model",
    "--run-id", FINAL_CONFIG["run_id"],
    "--device", FINAL_CONFIG["device"],
    "--run-root", str(RUN_ROOT),
    "--contract-path", str(CONTRACT_PATH),
    "--model-out-dir", str(MODEL_OUT_DIR),
    "--bundle-out-dir", str(BUNDLE_DIR),
    "--split-protocol", FINAL_CONFIG["split_protocol"],
    "--variant", FINAL_CONFIG["variant"],
    "--seed", str(FINAL_CONFIG["seed"]),
    "--monitor-target", FINAL_CONFIG["monitor_target"],
    "--alert-target", FINAL_CONFIG["alert_target"],
    "--normalization", FINAL_CONFIG["normalization"],
    "--metric-feature-mode", FINAL_CONFIG["metric_feature_mode"],
    "--epochs", str(FINAL_CONFIG["epochs"]),
    "--batch-size", str(FINAL_CONFIG["batch_size"]),
    "--patience", str(FINAL_CONFIG["patience"]),
    "--fp-budget", str(FINAL_CONFIG["fp_budget"]),
    "--quantization", FINAL_CONFIG["quantization"],
]

export_command = [
    sys.executable, "-m", "dashboard.export.bundle",
    "--mode", "protocol_fixed",
    "--protocol-run-dir", str(RUN_ROOT / RUN_ID),
    "--contract-path", str(CONTRACT_PATH),
    "--out-dir", str(BUNDLE_DIR),
    "--protocol-model-out-dir", str(MODEL_OUT_DIR),
    "--split-protocol", FINAL_CONFIG["split_protocol"],
    "--variant", FINAL_CONFIG["variant"],
    "--seed", str(FINAL_CONFIG["seed"]),
]

if FULL_RUN:
    run_command(train_command)
if EXPORT_AFTER_TRAIN:
    run_command(export_command)

RUN_DIR = find_run_dir(RUN_ROOT, RUN_ID, MODEL_OUT_DIR)
print("Активная директория результатов:", RUN_DIR)


In [ ]:
required_paths = {
    "comparison_metrics.csv": RUN_DIR / "comparison_metrics.csv",
    "train_manifest.json": RUN_DIR / "train_manifest.json",
    "alert_level_metrics.csv": RUN_DIR / "alert_level_metrics.csv",
    "workload_metrics.csv": RUN_DIR / "workload_metrics.csv",
    "bundle/dashboard_model_report.json": BUNDLE_DIR / "dashboard_model_report.json",
    "bundle/replay_bundle.csv.gz": BUNDLE_DIR / "replay_bundle.csv.gz",
}
display(file_status_table(required_paths))


**Локальный вывод.** После обучения доступны как аналитические артефакты (`comparison_metrics`, `train_manifest`), так и продуктовые (`bundle/*`) для демонстрации.


## 4. Кривые обучения и устойчивость по seed

**Цель раздела.** Показать динамику обучения по эпохам и стабильность результата между seed.


In [ ]:
train_manifest = safe_read_json(RUN_DIR / "train_manifest.json")
if isinstance(train_manifest, list) and train_manifest:
    manifest_df = pd.DataFrame(train_manifest)
    mask = (
        manifest_df["split_protocol"].astype(str).eq(FINAL_CONFIG["split_protocol"])
        & manifest_df["variant"].astype(str).eq(FINAL_CONFIG["variant"])
    )
    selected_runs = manifest_df.loc[mask].copy()
    display(selected_runs[["split_protocol", "variant", "seed", "best_epoch", "best_monitor", "train_monitor", "monitor_target"]])

    history_rows = []
    for _, row in selected_runs.iterrows():
        seed = int(row["seed"])
        best_epoch = int(row["best_epoch"])
        for rec in row.get("history", []):
            history_rows.append({"seed": seed, "best_epoch": best_epoch, **rec})
    history_df = pd.DataFrame(history_rows)
else:
    history_df = pd.DataFrame()

if history_df.empty:
    print("История обучения не найдена.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for seed, grp in history_df.groupby("seed"):
        grp = grp.sort_values("epoch")
        axes[0].plot(grp["epoch"], grp["train_loss"], marker="o", label=f"seed={seed}")
        axes[1].plot(grp["epoch"], grp["val_ap"], marker="o", label=f"seed={seed}")
    axes[0].set_title("Train loss по эпохам")
    axes[0].set_xlabel("Эпоха")
    axes[0].set_ylabel("train_loss")
    axes[1].set_title("Validation AP по эпохам")
    axes[1].set_xlabel("Эпоха")
    axes[1].set_ylabel("val_ap")
    axes[0].legend()
    axes[1].legend()
    plt.tight_layout()

    fig, ax = plt.subplots(figsize=(10, 4))
    for seed, grp in history_df.groupby("seed"):
        grp = grp.sort_values("epoch")
        ax.plot(grp["epoch"], grp["val_best_f1"], label=f"seed={seed} / val_best_f1")
        ax.plot(grp["epoch"], grp["val_fp_budget_f1"], linestyle="--", label=f"seed={seed} / val_fp_budget_f1")
        best_epoch = int(grp["best_epoch"].iloc[0])
        yv = grp.loc[grp["epoch"].eq(best_epoch), "val_best_f1"]
        if not yv.empty:
            ax.scatter([best_epoch], [yv.iloc[0]], s=60)
    ax.set_title("Validation F1 по эпохам и отметка лучшей эпохи")
    ax.set_xlabel("Эпоха")
    ax.set_ylabel("Значение метрики")
    ax.legend(ncol=2, fontsize=8)
    plt.tight_layout()


In [ ]:
comparison_all = safe_read_csv(RUN_DIR / "comparison_metrics.csv")
if comparison_all.empty:
    print("comparison_metrics.csv не найден")
else:
    seed_view = comparison_all[
        (comparison_all["split_protocol"].astype(str) == FINAL_CONFIG["split_protocol"])
        & (comparison_all["variant"].astype(str) == FINAL_CONFIG["variant"])
        & (comparison_all["runtime"].astype(str) == "torch_fp32")
        & (comparison_all["split"].astype(str) == "test")
        & (comparison_all["threshold_policy"].astype(str) == "validation_best_f1")
    ].copy()
    focus = seed_view[seed_view["target"].isin(["horizon_60s", "horizon_120s", "horizon_300s", "y_warning", "y_risk"])].copy()
    if focus.empty:
        print("Нет данных для анализа устойчивости по seed.")
    else:
        summary = focus.groupby("target").agg(
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            precision_mean=("precision", "mean"),
            recall_mean=("recall", "mean"),
        ).reset_index()
        display(summary.round(4))

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(summary["target"], summary["f1_mean"], yerr=summary["f1_std"].fillna(0), capsize=4)
        ax.set_title("Устойчивость F1 по seed (mean +/- std)")
        ax.set_xlabel("Target")
        ax.set_ylabel("F1")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=25)
        plt.tight_layout()


**Локальный вывод.** Кривые эпох показывают сходимость, а сводка по seed показывает воспроизводимость качества в фиксированном протоколе.


## 5. Результаты модели и сравнение с baseline

**Цель раздела.** Сравнить финальную модель и baseline по `F1`, `Precision`, `Recall`, `AP`, `ROC-AUC` на test.


In [ ]:
final_metrics = safe_read_csv(RUN_DIR / "comparison_metrics.csv")
if final_metrics.empty:
    print("comparison_metrics.csv не найден")
else:
    final_test = final_metrics[
        (final_metrics["split_protocol"].astype(str) == FINAL_CONFIG["split_protocol"])
        & (final_metrics["variant"].astype(str) == FINAL_CONFIG["variant"])
        & (pd.to_numeric(final_metrics["seed"], errors="coerce") == FINAL_CONFIG["seed"])
        & (final_metrics["runtime"].astype(str) == "torch_fp32")
        & (final_metrics["split"].astype(str) == "test")
        & (final_metrics["threshold_policy"].astype(str) == "validation_best_f1")
    ].copy()
    cols = ["target", "precision", "recall", "f1", "average_precision", "roc_auc", "false_positives", "missed_anomalies", "threshold"]
    display(final_test[cols].sort_values("target").round(4))


In [ ]:
baseline_metrics = safe_read_csv(BASELINE_DIR / "baseline_protocol_metrics.csv")
if baseline_metrics.empty or final_metrics.empty:
    print("Недостаточно данных для сравнения с baseline.")
else:
    baseline_best = baseline_metrics[
        (baseline_metrics["split_protocol"].astype(str) == FINAL_CONFIG["split_protocol"])
        & (baseline_metrics["split"].astype(str) == "test")
        & (baseline_metrics["threshold_policy"].astype(str) == "validation_best_f1")
    ].copy()
    baseline_best = baseline_best.sort_values(["target", "f1"], ascending=[True, False]).groupby("target", as_index=False).head(1)

    final_best = final_metrics[
        (final_metrics["split_protocol"].astype(str) == FINAL_CONFIG["split_protocol"])
        & (final_metrics["variant"].astype(str) == FINAL_CONFIG["variant"])
        & (pd.to_numeric(final_metrics["seed"], errors="coerce") == FINAL_CONFIG["seed"])
        & (final_metrics["runtime"].astype(str) == "torch_fp32")
        & (final_metrics["split"].astype(str) == "test")
        & (final_metrics["threshold_policy"].astype(str) == "validation_best_f1")
    ].copy()

    comp = pd.concat(
        [
            baseline_best.assign(model="Лучший baseline")[["target", "model", "precision", "recall", "f1", "average_precision", "roc_auc"]],
            final_best.assign(model="Efficient HADES")[["target", "model", "precision", "recall", "f1", "average_precision", "roc_auc"]],
        ],
        ignore_index=True,
    )
    display(comp.sort_values(["target", "model"]).round(4))

    plot_df = comp[comp["target"].isin(["horizon_60s", "horizon_120s", "horizon_300s", "y_warning", "y_risk"])].copy()
    if not plot_df.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        pivot = plot_df.pivot(index="target", columns="model", values="f1")
        pivot.plot(kind="bar", ax=ax)
        ax.set_title("Сравнение F1: Efficient HADES vs baseline")
        ax.set_xlabel("Target")
        ax.set_ylabel("F1")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=25)
        plt.tight_layout()


In [ ]:
# PR/ROC строятся по окнам test из test_predictions.csv.gz (если файл присутствует)
pred_path = RUN_DIR / "test_predictions.csv.gz"
pred = safe_read_csv(pred_path)
if pred.empty:
    print(f"Файл с окнами предсказаний не найден: {pred_path}")
else:
    pred = pred[
        (pred["split_protocol"].astype(str) == FINAL_CONFIG["split_protocol"])
        & (pred["variant"].astype(str) == FINAL_CONFIG["variant"])
        & (pd.to_numeric(pred["seed"], errors="coerce") == FINAL_CONFIG["seed"])
    ].copy()
    key_targets = ["horizon_60s", "horizon_120s", "horizon_300s"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    plotted = 0
    for target_name in key_targets:
        g = pred[pred["target_name"].astype(str) == target_name]
        if g.empty:
            continue
        y = pd.to_numeric(g["target"], errors="coerce").fillna(0).astype(int).to_numpy()
        s = pd.to_numeric(g["score"], errors="coerce").fillna(0).to_numpy()
        if len(np.unique(y)) < 2:
            continue
        pr, rc, _ = precision_recall_curve(y, s)
        fpr, tpr, _ = roc_curve(y, s)
        axes[0].plot(rc, pr, label=target_name)
        axes[1].plot(fpr, tpr, label=f"{target_name} (AUC={auc(fpr, tpr):.3f})")
        plotted += 1

    if plotted == 0:
        print("Для выбранных target не удалось построить PR/ROC (нет достаточного числа классов).")
    else:
        axes[0].set_title("PR-кривые")
        axes[0].set_xlabel("Recall")
        axes[0].set_ylabel("Precision")
        axes[0].legend()

        axes[1].set_title("ROC-кривые")
        axes[1].set_xlabel("False Positive Rate")
        axes[1].set_ylabel("True Positive Rate")
        axes[1].plot([0, 1], [0, 1], linestyle="--", linewidth=1)
        axes[1].legend()
        plt.tight_layout()


**Локальный вывод.** Таблицы и кривые показывают не только итоговый `F1`, но и форму разделения классов по score через PR/ROC, что важно для интерпретации поведения модели.


## 6. Анализ ошибок и alert-level trade-off

**Цель раздела.** Выделить типичные ошибки модели (FP/FN) и оценить компромисс `detected_incidents` vs `false_alerts`.


In [ ]:
pred = safe_read_csv(RUN_DIR / "test_predictions.csv.gz")
metrics_test = safe_read_csv(RUN_DIR / "comparison_metrics.csv")

if pred.empty or metrics_test.empty:
    print("Недостаточно данных для анализа FP/FN: нужны test_predictions и comparison_metrics.")
else:
    target_name = FINAL_CONFIG["monitor_target"]
    pred_t = pred[
        (pred["split_protocol"].astype(str) == FINAL_CONFIG["split_protocol"])
        & (pred["variant"].astype(str) == FINAL_CONFIG["variant"])
        & (pd.to_numeric(pred["seed"], errors="coerce") == FINAL_CONFIG["seed"])
        & (pred["target_name"].astype(str) == target_name)
    ].copy()

    thr_row = metrics_test[
        (metrics_test["split_protocol"].astype(str) == FINAL_CONFIG["split_protocol"])
        & (metrics_test["variant"].astype(str) == FINAL_CONFIG["variant"])
        & (pd.to_numeric(metrics_test["seed"], errors="coerce") == FINAL_CONFIG["seed"])
        & (metrics_test["runtime"].astype(str) == "torch_fp32")
        & (metrics_test["split"].astype(str) == "test")
        & (metrics_test["target"].astype(str) == target_name)
        & (metrics_test["threshold_policy"].astype(str) == "validation_best_f1")
    ]

    threshold = float(thr_row["threshold"].iloc[0]) if not thr_row.empty else 0.5
    pred_t["pred_label"] = (pd.to_numeric(pred_t["score"], errors="coerce").fillna(0) >= threshold).astype(int)
    pred_t["true_label"] = pd.to_numeric(pred_t["target"], errors="coerce").fillna(0).astype(int)

    fp = pred_t[(pred_t["pred_label"] == 1) & (pred_t["true_label"] == 0)].copy().sort_values("score", ascending=False).head(15)
    fn = pred_t[(pred_t["pred_label"] == 0) & (pred_t["true_label"] == 1)].copy().sort_values("score", ascending=True).head(15)

    print(f"Target: {target_name}, threshold={threshold:.4f}")
    print("Top FP:")
    display(fp[["run_id", "workload", "target_name", "score"]])
    print("Top FN:")
    display(fn[["run_id", "workload", "target_name", "score"]])

    workload_err = pred_t.groupby("workload").apply(
        lambda g: pd.Series(
            {
                "fp": int(((g["pred_label"] == 1) & (g["true_label"] == 0)).sum()),
                "fn": int(((g["pred_label"] == 0) & (g["true_label"] == 1)).sum()),
                "n": int(len(g)),
            }
        )
    ).reset_index()
    display(workload_err.sort_values(["fp", "fn"], ascending=False))


In [ ]:
alert = safe_read_csv(RUN_DIR / "alert_level_metrics.csv")
if alert.empty:
    print("alert_level_metrics.csv не найден")
else:
    alert_sel = alert[
        (alert["split_protocol"].astype(str) == FINAL_CONFIG["split_protocol"])
        & (alert["variant"].astype(str) == FINAL_CONFIG["variant"])
        & (pd.to_numeric(alert["seed"], errors="coerce") == FINAL_CONFIG["seed"])
        & (alert["target"].astype(str) == FINAL_CONFIG["alert_target"])
    ].copy()

    display(alert_sel.sort_values(["detected_incidents", "false_alerts"], ascending=[False, True]).head(20))

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(alert_sel["false_alerts"], alert_sel["detected_incidents"], alpha=0.8)
    for _, row in alert_sel.iterrows():
        ax.annotate(f"c={int(row['consecutive_windows'])}, cd={int(row['cooldown_windows'])}", (row["false_alerts"], row["detected_incidents"]), fontsize=8)
    ax.set_title("Alert-level trade-off: detected_incidents vs false_alerts")
    ax.set_xlabel("false_alerts")
    ax.set_ylabel("detected_incidents")
    plt.tight_layout()

    alert_sel["policy_score"] = alert_sel["detected_incidents"] - 0.5 * alert_sel["false_alerts"]
    best_policy = alert_sel.sort_values(["policy_score", "detected_incidents", "false_alerts"], ascending=[False, False, True]).head(1)
    print("Рекомендуемая policy (по правилу detected - 0.5 * false):")
    display(best_policy[["consecutive_windows", "cooldown_windows", "detected_incidents", "false_alerts", "missed_incidents", "policy_score"]])


**Локальный вывод.** Анализ FP/FN показывает, где модель ошибается по workload, а alert-level trade-off позволяет выбрать policy в терминах реальных инцидентов и ложных тревог.


## 7. Готовность к внедрению в dashboard

**Цель раздела.** Проверить полноту bundle и runtime-слоты `torch / onnx / int8` без подмены смысла качества.


In [ ]:
bundle_files = sorted([p for p in BUNDLE_DIR.glob("*") if p.is_file()])
print("Файлы bundle:")
for path in bundle_files:
    print(f"- {path.name} ({path.stat().st_size} байт)")

report = safe_read_json(BUNDLE_DIR / "dashboard_model_report.json")
runtime = report.get("runtime", {}) if isinstance(report, dict) else {}

runtime_table = pd.DataFrame(
    [
        {"runtime_slot": "torch", "status": runtime.get("torch_state", {}).get("status", "n/a"), "path": runtime.get("torch_state", {}).get("path", "n/a")},
        {"runtime_slot": "onnx_fp32", "status": runtime.get("onnx_fp32", {}).get("status", "n/a"), "path": runtime.get("onnx_fp32", {}).get("path", "n/a")},
        {"runtime_slot": "onnx_int8", "status": runtime.get("onnx_int8", {}).get("status", "n/a"), "path": runtime.get("onnx_int8", {}).get("path", "n/a")},
    ]
)
display(runtime_table)


**Локальный вывод.** Финальная модель подготовлена как единый академический и прикладной контур: обучение, проверка качества, анализ ошибок, выбор alert-policy и готовый dashboard bundle.

## Итог 03

Ноутбук формирует полный исследовательский цикл для финальной Protocol-Fixed Efficient HADES модели. Все ключевые выводы подтверждаются кодом, графиками и артефактами воспроизводимого запуска.
